# Tracing: MLflow Autolog and Harness Trace Events

This notebook demonstrates how the Custom Deep Research Lab captures execution traces
through two complementary systems:

1. **MLflow autolog tracing** — zero-code instrumentation of LangGraph agents,
   producing OpenTelemetry-compatible span trees viewable in the MLflow UI.
2. **Harness trace events** (`harness/trace.py`) — fine-grained, per-iteration
   events stored in PostgreSQL with session/layer/operation granularity.

Together these give you full visibility into *what* the agent did (MLflow spans)
and *how the harness loop evolved* (trace_events).

## 1. Load Environment

In [ ]:
import os
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "")
experiment = os.getenv("MLFLOW_EXPERIMENT_NAME", "deep-research-harness")
print(f"MLflow Tracking URI: {mlflow_uri or '(not set)'}")
print(f"Experiment: {experiment}")
if mlflow_uri:
    print("\u2705 Environment loaded")
else:
    print("\u26a0\ufe0f MLFLOW_TRACKING_URI not set \u2014 configure it in .env")

## 2. Connect to MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment(experiment)
exp = mlflow.get_experiment_by_name(experiment)
if exp:
    print(f"\u2705 Connected to experiment: {exp.name} (ID: {exp.experiment_id})")
else:
    print(f"\u2705 Created new experiment: {experiment}")

## 3. How Autolog Tracing Works

MLflow autolog tracing requires **zero code changes** to your LangGraph agents.
The entire setup is three lines in `backend/observability.py`:

```python
import mlflow
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.langchain.autolog()
```

Once enabled, MLflow automatically instruments:

| Component | What Gets Captured |
|-----------|-------------------|
| LangGraph execution | Full graph traversal path, node transitions |
| LLM calls | Model name, prompt, completion, token counts, latency |
| Tool invocations | Tool name, input arguments, output, duration |
| Agent decisions | Routing logic, conditional edges, state updates |

Each research session generates a single **trace** containing a tree of **spans** —
one span per operation.

## 4. Explore Traces in MLflow UI

Follow these steps to explore traces visually:

1. **Navigate to MLflow UI** — Open the RHOAI Dashboard and click the MLflow link
   (or use the route URL from `oc get route mlflow -n doc-research-lab`)
2. **Select the experiment** — Click on `deep-research-harness` in the left panel
3. **Click the Traces tab** — You will see a list of all recorded traces
4. **Click on a trace** — View the Summary (metadata), Details (span tree),
   and Timeline (Gantt chart of execution)
5. **Click individual spans** — Inspect tool inputs/outputs, LLM prompts and
   completions, token counts, and latency for each operation

**Tips:**
- Sort by duration to find slow research sessions
- Filter by status to locate errors
- Use the timeline view to identify sequential bottlenecks

## 5. Query Traces Programmatically

In [ ]:
traces = mlflow.search_traces(experiment_ids=[exp.experiment_id if exp else "0"])
if len(traces) > 0:
    print(f"Found {len(traces)} traces\n")
    for _, trace in traces.head(5).iterrows():
        print(f"  Trace: {trace.get('request_id', 'N/A')}")
        print(f"  Status: {trace.get('status', 'N/A')}")
        print(f"  Duration: {trace.get('execution_duration', 'N/A')}ms")
        print()
else:
    print("No traces found yet \u2014 run a research query first")
print("\u2705 Trace query complete")

## 6. Span Structure

A typical research session produces the following span tree:

```
[LangGraph] research_session
  ├── [normalize]
  ├── [plan] → ChatOpenAI (LLM call)
  ├── [execute]
  │     ├── semantic_search (MCP tool)
  │     ├── web_search (SearXNG tool)
  │     └── draft_report → ChatOpenAI
  ├── [verify] → ChatOpenAI (LLM-as-Judge)
  ├── [observe]
  └── [finalize]
```

Each span records:
- **Start/end timestamps** — for duration calculations
- **Input/output** — full payloads (prompts, tool args, responses)
- **Attributes** — model name, token counts, status codes
- **Parent span ID** — linking the full call hierarchy

## 7. Harness Trace Events

In addition to MLflow spans, the harness records its own trace events to PostgreSQL
via `harness/trace.py`. These events map to harness loop phases:

| Harness Layer | MLflow Span | trace_events.layer | trace_events.operation |
|---------------|-------------|--------------------|-----------------------|
| Plan | `[plan]` | `context` | `generate_plan`, `rewrite_query` |
| Execute | `[execute] > semantic_search` | `tool` | `semantic_search`, `web_search` |
| Execute | `[execute] > draft_report` | `execution` | `draft_report`, `synthesize` |
| Verify | `[verify]` | `verification` | `quality_check`, `citation_check` |
| Observe | `[observe]` | `observability` | `record_metrics`, `check_threshold` |

The key difference: MLflow spans capture the *call tree* (what called what),
while harness trace events capture the *iteration history* (how quality evolved
across passes).

Query harness trace events for a specific session to see the iteration-by-iteration
breakdown alongside MLflow trace data.

In [ ]:
try:
    import psycopg2
    conn = psycopg2.connect(
        host=os.getenv("PGVECTOR_HOST", "localhost"),
        port=os.getenv("PGVECTOR_PORT", "5432"),
        dbname=os.getenv("PGVECTOR_DB", "doc_research"),
        user=os.getenv("PGVECTOR_USER", "postgres"),
        password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
    )
    cur = conn.cursor()

    cur.execute("""
        SELECT session_id FROM trace_events
        GROUP BY session_id
        ORDER BY MAX(timestamp) DESC
        LIMIT 1
    """)
    row = cur.fetchone()

    if row:
        sid = row[0]
        print(f"Latest session: {sid}\n")

        cur.execute("""
            SELECT iteration, layer, operation, latency_ms, tokens_used, success,
                   failure_category
            FROM trace_events
            WHERE session_id = %s
            ORDER BY timestamp
        """, (sid,))
        events = cur.fetchall()

        print(f"{'Iter':>4} {'Layer':<15} {'Operation':<20} {'Latency':>8} {'Tokens':>7} {'OK':>3} {'Failure'}")
        print("-" * 80)
        for e in events:
            fail = e[6] or ""
            ok = "\u2705" if e[5] else "\u274c"
            print(f"{e[0]:>4} {e[1]:<15} {e[2]:<20} {e[3]:>6}ms {e[4]:>7} {ok:>3} {fail}")
    else:
        print("No trace events found \u2014 run a research query first")

    cur.close()
    conn.close()
    print("\n\u2705 Harness trace query complete")
except ImportError:
    print("\u26a0\ufe0f psycopg2 not installed \u2014 pip install psycopg2-binary")
except Exception as e:
    print(f"\u26a0\ufe0f PostgreSQL unavailable ({e}) \u2014 start with 'make dev-up'")

## 8. Side-by-Side: MLflow Spans vs Harness Events

Both views complement each other:

```
MLflow UI (Traces tab)              Harness trace_events (PostgreSQL)
\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500              \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
\u2022 One trace per request             \u2022 Multiple events per iteration
\u2022 Span tree (call hierarchy)         \u2022 Flat events (iteration timeline)
\u2022 Auto-captured (zero code)          \u2022 Explicitly recorded by harness
\u2022 LLM prompts + completions          \u2022 Layer, operation, failure category
\u2022 Token counts per span              \u2022 Cumulative tokens per session
\u2022 Visual timeline (Gantt chart)       \u2022 SQL-queryable for aggregation
```

**When to use which:**
- **Debugging a single request** \u2192 MLflow UI span tree (see exact prompts/outputs)
- **Analyzing quality evolution** \u2192 Harness events (iteration-by-iteration scores)
- **Finding failure patterns** \u2192 Harness events (query by `failure_category`)
- **Performance profiling** \u2192 MLflow timeline (Gantt chart shows bottlenecks)

## 9. Summary

In [ ]:
print("\u2705 MLflow tracing walkthrough complete")
print()
print("What we covered:")
print("  1. Connected to MLflow tracking server")
print("  2. Understood autolog tracing (zero code changes)")
print("  3. Explored traces via the MLflow UI")
print("  4. Queried traces programmatically")
print("  5. Reviewed the span tree structure for research sessions")
print("  6. Mapped harness trace events to MLflow spans")
print("  7. Compared MLflow spans vs harness events side-by-side")
print()
print("Next: 4_agent_evaluation.ipynb \u2014 Agent evaluation with MLflow GenAI scorers")